# 원본 RMT Remember · Colab T4 (1 → 2쌍)

수정본을 `/content/GRT`에 pull한 뒤 아래 셀을 순서대로 실행합니다.

- 원본 backbone·memory wrapper·데이터·내부 loss, D128 / 4층 / memory32 / FP32.
- microbatch2,048 × accumulation1. 1쌍 최대500 → 2쌍 최대2,500 update.
- 원본 데이터 규모: train1,000,000 / validation1,000 / test10,000.
- 원본 B512의2,000/10,000 update와 같은 명목 샘플 노출량입니다. 99% 생성 정확도에서 조기 종료합니다.
- 예상 peak 약9.65GiB, 현재 처리량 환산 학습 약51분 + 평가·저장 시간. 실제 속도는 로그로 확인합니다.
- 새 RUN_DIR을 사용합니다. 같은 설정·경로로 재실행하면 자동 재개합니다.
- Drive에서 각 stage의 `metrics.txt`로 진행을 확인하고, 최종 `summary.txt`와 `evaluation.txt`를 읽으면 됩니다.


In [ ]:
from pathlib import Path
import sys, subprocess, torch
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = Path('/content/GRT')
RUN_DIR = Path('/content/drive/MyDrive/GRT/runs/rmt_author_t4_b2048_samples_seed54')
assert (REPO_DIR / 'configs/rmt_author_t4.yaml').exists(), '수정본을 pull한 checkout이 필요합니다.'
assert torch.cuda.is_available(), 'T4 GPU runtime을 선택하세요.'
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
if installation.returncode:
    print(installation.stdout)
    installation.check_returncode()
sys.path.insert(0, str(REPO_DIR / 'src'))
from grt.config import load_config
cfg = load_config([
    str(REPO_DIR / 'configs/rmt_author_reference.yaml'),
    str(REPO_DIR / 'configs/rmt_author_t4.yaml'),
], output_dir=RUN_DIR)
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Effective batch:', cfg.training.batch_size * cfg.training.grad_accum_steps)
print('Run:', RUN_DIR)


In [ ]:
# 새 학습 또는 동일 설정의 last.pt에서 자동 재개.
subprocess.run([
    sys.executable, '-u', 'scripts/train.py', '--config',
    'configs/rmt_author_reference.yaml', 'configs/rmt_author_t4.yaml',
    '--output-dir', str(RUN_DIR), '--device', 'cuda',
], cwd=REPO_DIR, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt

summary = json.loads((RUN_DIR / 'summary.json').read_text())
evaluation = json.loads((RUN_DIR / 'evaluation.json').read_text())
print(json.dumps(summary, indent=2))
print(json.dumps(evaluation, indent=2))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
offset = 0
for index, stage in enumerate(summary['stages'], 1):
    stage_dir = RUN_DIR / f"stage_{index:02d}_pairs{stage['num_pairs']}_key{stage['key_size']}"
    metrics = stage_dir / 'metrics.jsonl'
    rows = [json.loads(line) for line in metrics.read_text().splitlines() if line.strip()] if metrics.exists() else []
    steps = [offset + row['step'] for row in rows]
    label = f"{stage['num_pairs']} pairs"
    axes[0].plot(steps, [row['validation']['fixed_length']['loss'] for row in rows], label=label)
    axes[1].plot(steps, [row['validation']['fixed_length']['exact_match'] for row in rows], label=label)
    offset += stage['step']
axes[0].set_ylabel('fixed-length validation CE (value + EOS)')
axes[1].set_ylabel('generated exact match (value + EOS)')
axes[1].axhline(.99, color='gray', linestyle='--')
axes[1].set_ylim(0, 1.03)
for ax in axes:
    ax.set_xlabel('total optimizer updates')
    ax.grid(alpha=.2)
    ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'convergence.png', dpi=150)
plt.show()
print('Drive 확인: summary.txt, evaluation.txt 및 각 stage의 metrics.txt')
print('공유: summary.json, evaluation.json, convergence.png 및 각 stage의 metrics.jsonl, resolved_config.yaml, metadata.json')
